In [ ]:
## RAGAS is a framework for evaluating the RAG pipeline.

## Metrics:
# 1. Context Relevancy: Measures how relevant the retrieved context is to the question asked.
#    It filters out irrelevant information from the retrieved context.
#    Calculated as: number of relevant sentences in context / total sentences in context.

# 2. Context Precision: Measures the signal-to-noise ratio of the retrieved context.
#    It evaluates whether the relevant chunks are ranked higher than irrelevant ones.
#    Calculated as: mean of precision@k for each relevant chunk in the ranked retrieved context,
#    where precision@k = number of relevant chunks in top-k / k.

# 3. Context Recall: Measures how much of the ground truth is captured in the retrieved context.
#    It checks if all the necessary information to answer the question was retrieved.
#    Calculated as: Recall@K = number of ground truth sentences attributable to context / total sentences in ground tru th.

# 4. Faithfulness: Measures how factually consistent the generated answer is with the retrieved context.
#    It ensures the answer does not contain information not present in the context (no hallucinations).
#    Calculated as: number of answer statements that can be inferred from context / total statements in answer.

# 5. Answer Relevancy: Measures how relevant the generated answer is to the original question.
#    It penalizes answers that are incomplete or contain redundant information.
#    Calculated as: mean cosine similarity between the original question and n questions
#    generated from the answer, using embeddings to capture semantic similarity.

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_postgres import PGVector
from langchain_community.document_loaders import DirectoryLoader
import os
from dotenv import load_dotenv

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents.base import Document
from dotenv import load_dotenv
from langchain_chroma import Chroma
import os
from euriai.langchain import create_chat_model
import time
from euriai.langchain import EuriaiEmbeddings

app_dir = os.path.join(os.getcwd(), "app")
load_dotenv(os.path.join(app_dir, ".env"))

api_key = os.getenv("key")

chat_model = create_chat_model(api_key=api_key, model="gpt-4.1-nano", temperature=0.7)
model = chat_model

embeddings = EuriaiEmbeddings(
    api_key=api_key,
    model="text-embedding-3-small"
)

In [2]:
loader = DirectoryLoader("./data", glob="**/*.txt")
docs = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,
    chunk_overlap=20,
    length_function=len,
    is_separator_regex=False,
)
chunks = text_splitter.split_documents(docs)

libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.
libmagic is unavailable but assists in filetype detection. Please consider installing libmagic for better results.


In [3]:
chunks[0]

Document(metadata={'source': 'data\\food.txt'}, page_content='margherita pizza; $12; classic with tomato, mozzarella, and basil; main dish\n\nspaghetti carbonara; $15; creamy pasta with pancetta and parmesan; main dish\n\nbruschetta; $8; toasted bread with tomato, garlic, and olive oil; appetizer\n\ncaprese salad; $10; fresh tomatoes, mozzarella, and basil; salad')

In [4]:
# RAGAS expects a file_name dict as key
for document in chunks:
    document.metadata["file_name"] = document.metadata["source"]

In [5]:
chunks[0]

Document(metadata={'source': 'data\\food.txt', 'file_name': 'data\\food.txt'}, page_content='margherita pizza; $12; classic with tomato, mozzarella, and basil; main dish\n\nspaghetti carbonara; $15; creamy pasta with pancetta and parmesan; main dish\n\nbruschetta; $8; toasted bread with tomato, garlic, and olive oil; appetizer\n\ncaprese salad; $10; fresh tomatoes, mozzarella, and basil; salad')

In [15]:
from ragas.testset import TestsetGenerator
from ragas.testset.evolutions import simple, reasoning, multi_context

generator = TestsetGenerator.from_langchain(
    embedding_model=embeddings, generator_llm=model, critic_llm=model
)

testset = generator.generate_with_langchain_docs(
    chunks,
    test_size=8,
    distributions={simple: 0.5, reasoning: 0.25, multi_context: 0.25},
)

TypeError: TestsetGenerator.from_langchain() got an unexpected keyword argument 'generator_llm'

In [ ]:
testset.to_pandas()

In [ ]:
from langchain_openai.embeddings import OpenAIEmbeddings

from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI

embedding = OpenAIEmbeddings()
model = ChatOpenAI(model="gpt-4o-mini")

vectorstore = Chroma.from_documents(chunks, embedding)
retriever = vectorstore.as_retriever()

In [ ]:
from langchain_core.prompts import PromptTemplate

template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = PromptTemplate(template=template, input_variables=["context", "question"])

In [ ]:
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

In [ ]:
# questions = testset.to_pandas()["question"].to_list()
# ground_truth = testset.to_pandas()["ground_truth"].to_list()

import pandas as pd

df = pd.read_csv("./questions_answers/qa.csv", delimiter=";")
questions = df["question"].tolist()
ground_truth = df["ground_truth"].tolist()

In [ ]:
ground_truth

In [ ]:
from datasets import Dataset

data = {"question": [], "answer": [], "contexts": [], "ground_truth": ground_truth}

for query in questions:
    data["question"].append(query)
    data["answer"].append(rag_chain.invoke(query))
    data["contexts"].append(
        [doc.page_content for doc in retriever.invoke(query)]
    )

dataset = Dataset.from_dict(data)

In [ ]:
first_entry = {
    "question": data["question"][0],
    "answer": data["answer"][0],
    "contexts": data["contexts"][0],
    "ground_truth": data["ground_truth"][0],
}
first_entry

In [ ]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_relevancy,
    context_recall,
    context_precision,
)

result = evaluate(
    dataset=dataset,
    metrics=[
        context_relevancy,
        context_precision,
        context_recall,
        faithfulness,
        answer_relevancy,
    ],
    llm=model,
    embeddings=embedding
)

In [ ]:
result.to_pandas()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

df = result.to_pandas()

heatmap_data = df[
    [
        "context_relevancy",
        "context_precision",
        "context_recall",
        "faithfulness",
        "answer_relevancy",
    ]
]

cmap = LinearSegmentedColormap.from_list("green_red", ["red", "green"])

plt.figure(figsize=(10, 8))
sns.heatmap(heatmap_data, annot=True, fmt=".2f", linewidths=0.5, cmap=cmap)

plt.yticks(ticks=range(len(df["question"])), labels=df["question"], rotation=0)

plt.show()